# Phishing Website Detection Using Machine Learning

**Student:** A. Ashwell  
**Programme:** B.Tech Cyber Security  
**Project:** IBM SkillsBuild Cyber Security 2026

This educational project extracts simple lexical features from a URL and trains a Random Forest classifier. It is a learning demonstration, not a production security product. A model trained on the small fallback sample below is not reliable for real-world decisions.

## 1. Install and import libraries
If needed, run `pip install -r requirements.txt` in your environment before running this notebook.

In [ ]:
import re
import ipaddress
from urllib.parse import urlparse
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

RANDOM_STATE = 42
print('Libraries imported successfully.')

## 2. Create URL features
The function below looks only at the URL text. It does not visit websites or verify whether a site is malicious.

In [ ]:
def extract_features(url):
    url = str(url).strip()
    candidate = url if re.match(r'^[a-zA-Z][a-zA-Z0-9+.-]*://', url) else 'http://' + url
    parsed = urlparse(candidate)
    host = parsed.hostname or ''
    try:
        ipaddress.ip_address(host)
        uses_ip = 1
    except ValueError:
        uses_ip = 0
    return {
        'url_length': len(url),
        'host_length': len(host),
        'dot_count': url.count('.'),
        'hyphen_count': url.count('-'),
        'at_count': url.count('@'),
        'question_count': url.count('?'),
        'equals_count': url.count('='),
        'slash_count': url.count('/'),
        'has_https': int(parsed.scheme.lower() == 'https'),
        'uses_ip_address': uses_ip,
        'has_suspicious_word': int(any(w in url.lower() for w in ['login', 'verify', 'update', 'secure', 'account', 'password', 'banking', 'confirm']))
    }

example_url = 'https://example.com'
pd.DataFrame([extract_features(example_url)])

## 3. Load a dataset or use a small demonstration dataset
For a meaningful evaluation, place a labelled CSV named `phishing_site_urls.csv` beside this notebook. It should contain columns named `URL` and `Label`, with labels such as `good`/`bad` or `legitimate`/`phishing`.

If no CSV is found, the notebook uses a tiny hand-written demonstration dataset so the notebook can run from start to finish. The fallback data is only for demonstrating the workflow; its scores must not be treated as real-world performance.

In [ ]:
dataset_path = 'phishing_site_urls.csv'
try:
    raw = pd.read_csv(dataset_path)
    if not {'URL', 'Label'}.issubset(raw.columns):
        raise ValueError("CSV must contain columns named 'URL' and 'Label'.")
    data = raw[['URL', 'Label']].dropna().copy()
    print(f'Loaded {len(data)} rows from {dataset_path}.')
except FileNotFoundError:
    print('CSV not found. Using small demo data; do not use its evaluation score as evidence of real-world accuracy.')
    demo = [
        ('https://www.wikipedia.org', 'good'),
        ('https://www.python.org', 'good'),
        ('https://www.microsoft.com', 'good'),
        ('https://www.sona.ac.in', 'good'),
        ('https://www.openai.com', 'good'),
        ('https://docs.python.org/3/', 'good'),
        ('https://www.nationalgeographic.com', 'good'),
        ('https://www.github.com', 'good'),
        ('http://192.0.2.10/login/verify', 'bad'),
        ('http://account-verify-example.test/login?user=a', 'bad'),
        ('http://secure-update-example.test/confirm?account=1', 'bad'),
        ('http://banking-login-example.test/verify/password', 'bad'),
        ('http://example.test@192.0.2.20/login', 'bad'),
        ('http://password-reset-example.test/account/update', 'bad'),
        ('http://verify-account-example.test/secure/login', 'bad'),
        ('http://192.0.2.30/confirm?banking=1', 'bad'),
    ]
    data = pd.DataFrame(demo, columns=['URL', 'Label'])

data['Label'] = data['Label'].astype(str).str.lower().str.strip()
label_map = {
    'good': 0, 'legitimate': 0, 'benign': 0, 'safe': 0,
    'bad': 1, 'phishing': 1, 'malicious': 1, 'unsafe': 1
}
data['target'] = data['Label'].map(label_map)
data = data.dropna(subset=['URL', 'target']).copy()
data['target'] = data['target'].astype(int)
print(data['target'].value_counts().rename(index={0:'legitimate', 1:'phishing'}))
data.head()

## 4. Prepare features and train the model
The split is stratified when possible. With the small fallback dataset, metrics are only a software demonstration and will vary with the data.

In [ ]:
X = pd.DataFrame([extract_features(u) for u in data['URL']])
y = data['target']

if len(data) >= 8 and y.nunique() == 2 and y.value_counts().min() >= 2:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
    )
else:
    X_train, X_test, y_train, y_test = X, X, y, y
    print('Warning: dataset is too small for a useful independent test set.')

model = RandomForestClassifier(n_estimators=150, random_state=RANDOM_STATE, class_weight='balanced')
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print('Accuracy on this dataset split:', round(accuracy_score(y_test, predictions), 3))
print('\nClassification report:\n', classification_report(y_test, predictions, zero_division=0))
print('Confusion matrix:\n', confusion_matrix(y_test, predictions))

## 5. Predict a URL
This predicts from URL-text features only. A result of “Potential phishing” is a warning to investigate, not proof that a website is malicious. Do not enter passwords or other sensitive information into suspicious websites.

In [ ]:
def predict_url(url):
    features = pd.DataFrame([extract_features(url)])
    label = int(model.predict(features)[0])
    probability = float(model.predict_proba(features)[0][1])
    verdict = 'Potential phishing — verify carefully' if label == 1 else 'Likely legitimate based on this model'
    return {'url': url, 'prediction': verdict, 'model_phishing_score': round(probability, 3)}

predict_url('http://account-verify-example.test/login?user=a')

## 6. Conclusion and limitations
- The notebook demonstrates feature extraction and supervised classification for URL screening.
- Accuracy depends on the quality, size, freshness and representativeness of the labelled dataset.
- Attackers can change URLs, and legitimate websites can have unusual-looking URLs.
- This project does not visit URLs, scan page content, check certificates, or guarantee safety.
- For a report with meaningful results, use a reputable labelled dataset, document its source and licence, remove duplicates, and evaluate on a separate test set.